# CLAP Playground (LAION)

Ce notebook permet de tester CLAP sur les morceaux du dossier `Songs/`:
- embeddings audio
- embeddings texte
- similarité texte ↔ audio
- génération d'une mini playlist par prompt

In [5]:
# Si besoin, décommentez la ligne suivante pour installer les dépendances
# %pip install -q transformers torch torchaudio librosa soundfile

In [6]:
from pathlib import Path
import numpy as np
import torch
import librosa
from transformers import ClapProcessor, ClapModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [7]:
songs_dir = Path('Songs')
audio_exts = {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
audio_files = sorted([p for p in songs_dir.iterdir() if p.suffix.lower() in audio_exts])

print(f'{len(audio_files)} fichier(s) audio trouvé(s) dans {songs_dir}/')
for i, p in enumerate(audio_files, 1):
    print(f'{i:02d}. {p.name}')

2 fichier(s) audio trouvé(s) dans Songs/
01. Dr. G - RENEGADE MASTER [regroove].mp3
02. ♥ PREMIERE ♥ Prosekko Papi - Come Into My House.mp3


In [9]:
if 'model_id' not in locals():
    model_id = 'laion/clap-htsat-fused'
    processor = ClapProcessor.from_pretrained(model_id)
    model = ClapModel.from_pretrained(model_id).to(device)
    model_id

Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

In [35]:
def load_audio_mono_48k(path):
    # CLAP attend généralement de l'audio mono à 48 kHz
    wav, sr = librosa.load(path, sr=48000, mono=True)
    return wav

def l2_normalize(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)

def build_audio_embeddings(paths, batch_size=4):
    if len(paths) == 0:
        return np.empty((0, model.config.projection_dim), dtype=np.float32)

    all_embs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            batch_wavs = [load_audio_mono_48k(p) for p in batch_paths]
            inputs = processor(audio=batch_wavs, sampling_rate=48000, return_tensors='pt', padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            audio_out = model.get_audio_features(**inputs)
            audio_tensor = audio_out.pooler_output
            all_embs.append(audio_tensor.detach().cpu().numpy())
    return np.vstack(all_embs)

audio_embs = build_audio_embeddings(audio_files)
audio_embs = l2_normalize(audio_embs)
audio_embs.shape

(2, 512)

In [37]:
def text_embeddings(prompts):
    model.eval()
    with torch.no_grad():
        inputs = processor(text=prompts, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        text_out = model.get_text_features(**inputs)

        if hasattr(text_out, 'text_embeds'):
            text_tensor = text_out.text_embeds
        elif hasattr(text_out, 'pooler_output'):
            text_tensor = text_out.pooler_output
        else:
            text_tensor = text_out

        txt = text_tensor.detach().cpu().numpy()
    return l2_normalize(txt)

def rank_tracks_for_prompt(prompt, top_k=5):
    t = text_embeddings([prompt])[0]
    scores = audio_embs @ t
    idx = np.argsort(-scores)[:top_k]
    return [(audio_files[i].name, float(scores[i])) for i in idx]

test_prompt = 'energetic club track with a strong bassline'
results = rank_tracks_for_prompt(test_prompt, top_k=min(5, len(audio_files)))

print('Prompt:', test_prompt)
for name, score in results:
    print(f'{score: .4f}  |  {name}')

Prompt: energetic club track with a strong bassline
 0.2269  |  Dr. G - RENEGADE MASTER [regroove].mp3
 0.1076  |  ♥ PREMIERE ♥ Prosekko Papi - Come Into My House.mp3


In [32]:
# Exemples de prompts à tester
prompts = [
    'house music with a groovy beat',
    'hard and aggressive electronic beat',
    'melodic and emotional dance track'
]

for p in prompts:
    print('\n' + '=' * 80)
    print('PROMPT:', p)
    for name, score in rank_tracks_for_prompt(p, top_k=min(3, len(audio_files))):
        print(f'{score: .4f}  |  {name}')


PROMPT: house music with a groovy beat
 0.1820  |  Dr. G - RENEGADE MASTER [regroove].mp3
 0.0305  |  ♥ PREMIERE ♥ Prosekko Papi - Come Into My House.mp3

PROMPT: hard and aggressive electronic beat
 0.2746  |  Dr. G - RENEGADE MASTER [regroove].mp3
 0.0732  |  ♥ PREMIERE ♥ Prosekko Papi - Come Into My House.mp3

PROMPT: melodic and emotional dance track
 0.0124  |  Dr. G - RENEGADE MASTER [regroove].mp3
-0.0558  |  ♥ PREMIERE ♥ Prosekko Papi - Come Into My House.mp3
